In [13]:
import os
import sys

from osgeo import gdal

import cv2
import numpy as np 
import skimage.filters
import skimage.color
import skimage.io
import matplotlib.pyplot as plt
import shapely
from shapely.geometry.polygon import LinearRing, Polygon, LineString, Point
from shapely.geometry import MultiPolygon
import pandas as pd
from area import area
from shapely.ops import cascaded_union
from PIL import Image
from keras.layers import *
from pyproj import Transformer

In [14]:
import keras

from tensorflow.keras.models import load_model
model_loaded = load_model('modeldata_foliage/foliage-detector-v11_102+OUTDATED.h5')

keras.utils.plot_model(model_loaded, "architecture.png", show_shapes=True)

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [ ]:
data_dir='modeldata_foliage'
labels = ['foliage_COMBINED_original+OUTDATED', 'other']
img_size = 224  #224
def get_data(data_dir):
    data = [] 
    for label in labels: 
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        
        cnt=0
        for img in os.listdir(path):
            #if cnt<=1000:
            try:
                img_arr = cv2.imread(os.path.join(path, img))[...,::-1] #convert BGR to RGB format
                resized_arr = cv2.resize(img_arr, (img_size, img_size)) # Reshaping images to preferred size
                data.append([resized_arr, class_num])
                cnt=cnt+1
            except Exception as e:
                print(e)
        print(label,'   --   ',cnt)
    return np.array(data)

In [ ]:
from sklearn.model_selection import train_test_split

alldata=get_data(data_dir)
train, val = train_test_split(alldata, test_size=0.2, random_state=7)
k,l,m = [],[],[]

for i in alldata:
    if(i[1] == 0):
        k.append("foliage")
    else:
        k.append("other")

for i in train:
    if(i[1] == 0):
        l.append("foliage_train")
    else:
        l.append("other_train")

for i in val:
    if(i[1] == 0):
        m.append("foliage_val")
    else:
        m.append("other_val")

"""
sns.set_style('darkgrid')
sns.countplot(l)
"""
df1=pd.DataFrame(data=k,columns=['data'])
df2=pd.DataFrame(data=l,columns=['data'])
df3=pd.DataFrame(data=m,columns=['data'])

In [ ]:
sns.set_style('darkgrid')
sns.countplot(df1, x='data')
#sns.countplot(df2, x='data')
#sns.countplot(df3, x='data')

In [ ]:
x_train = []
y_train = []
x_val = []
y_val = []

for feature, label in train:
    x_train.append(feature)
    y_train.append(label)

for feature, label in val:
    x_val.append(feature)
    y_val.append(label)

# Normalize the data
x_train = np.array(x_train) / 255
x_val = np.array(x_val) / 255

x_train.reshape(-1, img_size, img_size, 1)
y_train = np.array(y_train)

x_val.reshape(-1, img_size, img_size, 1)
y_val = np.array(y_val)

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('modeldata_foliage/foliage-detector-v11_102+OUTDATED.h5')

In [ ]:
opt = Adam(lr=0.000001)
model.compile(optimizer = opt , loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) , metrics = ['accuracy'])
history = model.fit(x_train,y_train,epochs = 500 , validation_data = (x_val, y_val))

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(500)

plt.figure(figsize=(15, 15))
plt.subplot(2, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
#predictions = (model.predict(x_val) > 0.5).astype("int32")
predict_x = model.predict(x_val)
classes_x=np.argmax(predict_x,axis=1)
print(classification_report(y_val, classes_x, target_names = ['Foliage (Class 0)','Other (Class 1)']))